# CareCompanion: Skills, Memory & Dreams for Health Agents
## Powered by Gemma 4 26B — Replicating Anthropic's Managed Agent Primitives

**[Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon) — Health & Sciences Track**

This notebook demonstrates CareCompanion, a personal wellness assistant that helps elderly patients manage daily health routines. It uses **Gemma 4 26B-A4B** to power three agent primitives:

1. **Skills** — Reusable expertise modules loaded on-demand (progressive disclosure)
2. **Memory** — Persistent, SHA256-versioned health records across sessions
3. **Dreams** — Offline memory consolidation that discovers clinical insights from session transcripts

### What We'll Demonstrate

| Step | What | Why |
|------|------|-----|
| 1 | Explore the skill system | Show how expertise is loaded on-demand |
| 2 | Browse memory stores | Show persistent health records |
| 3 | Run a care session with Gemma 4 | Live agent with 15 tools, skills, and memory |
| 4 | Examine the session trace | Show what the agent recorded |
| 5 | Run dream consolidation | Offline analysis discovers clinical patterns |
| 6 | Review dream insights | Show medically actionable findings |

---

**GitHub**: [EvolvingAgentsLabs/skillos_x_mobile](https://github.com/EvolvingAgentsLabs/skillos_x_mobile)  
**Video**: [YouTube Demo](https://youtu.be/8ho3SpY-rDU)

## Setup

Clone the repository and install dependencies.

> **API Key**: This notebook requires an [OpenRouter API key](https://openrouter.ai/keys) to call Gemma 4. Add it as a Kaggle Secret named `OPENROUTER_API_KEY` (Add-ons > Secrets).

In [ ]:
# Install Node.js 20+ (Kaggle has Node.js pre-installed, but let's ensure the right version)
import subprocess, os, shutil

# Check Node.js version
node_version = subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
print(f"Node.js version: {node_version}")

npm_version = subprocess.run(['npm', '--version'], capture_output=True, text=True).stdout.strip()
print(f"npm version: {npm_version}")

In [ ]:
%%bash
# Clone the repository
cd /kaggle/working
if [ ! -d "skillos_x_mobile" ]; then
    git clone https://github.com/EvolvingAgentsLabs/skillos_x_mobile.git
fi
cd skillos_x_mobile
npm install --silent 2>&1 | tail -3
echo "\n--- Setup complete ---"

In [ ]:
import os

PROJECT_DIR = '/kaggle/working/skillos_x_mobile'
os.chdir(PROJECT_DIR)

# Load API key from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    api_key = secrets.get_secret('OPENROUTER_API_KEY')
    os.environ['OPENROUTER_API_KEY'] = api_key
    print('OpenRouter API key loaded from Kaggle Secrets')
except Exception:
    # Fallback: set manually if running outside Kaggle
    if 'OPENROUTER_API_KEY' not in os.environ:
        print('WARNING: Set OPENROUTER_API_KEY manually if running outside Kaggle')
    else:
        print('OpenRouter API key found in environment')

---
## 1. Skills — Progressive Disclosure

Skills are reusable expertise modules stored as markdown files. The agent only sees skill **names and descriptions** in its system prompt (~100 tokens/skill). When it needs a skill, it calls `load_skill(name)` to retrieve the full instructions.

This is functionally equivalent to [Anthropic's Skills API](https://docs.anthropic.com/en/docs/agents-and-tools/managed-agents#skills) — but implemented entirely with open-weight Gemma 4.

In [ ]:
import glob, re
from IPython.display import display, Markdown, HTML

skill_files = sorted(glob.glob(f'{PROJECT_DIR}/skills/*.skill.md'))

print(f'Found {len(skill_files)} skills:\n')

skills_table = '| Skill | Description | Instructions (tokens) |\n|-------|-------------|----------------------|\n'

for sf in skill_files:
    with open(sf) as f:
        content = f.read()
    
    # Parse YAML frontmatter
    fm_match = re.match(r'^---\s*\n([\s\S]*?)\n---\s*\n([\s\S]*)$', content)
    if fm_match:
        frontmatter, instructions = fm_match.group(1), fm_match.group(2)
        name = re.search(r'^name:\s*(.+)', frontmatter, re.M)
        desc = re.search(r'^description:\s*(.+)', frontmatter, re.M)
        name = name.group(1).strip() if name else os.path.basename(sf)
        desc = desc.group(1).strip() if desc else ''
        # Rough token estimate
        token_est = len(instructions.split())
        skills_table += f'| `{name}` | {desc} | ~{token_est} |\n'

display(Markdown(skills_table))
display(Markdown('\n> **Level 1** (system prompt): Only the name + description above are included (~400 tokens total).  \n'
                 '> **Level 2** (on demand): Full instructions are loaded only when `load_skill()` is called.'))

In [ ]:
# Let's look at the medication-reminder skill in detail
with open(f'{PROJECT_DIR}/skills/medication-reminder.skill.md') as f:
    content = f.read()

display(Markdown('### Skill: `medication-reminder` (Full Instructions)\n\n'
                 'This is what the agent receives when it calls `load_skill("medication-reminder")`:\n'))
display(Markdown(content))

---
## 2. Memory — Persistent Health Records

Memory stores are directories of versioned markdown documents. Every write creates a SHA256-versioned backup.

This is functionally equivalent to [Anthropic's Memory API](https://docs.anthropic.com/en/docs/agents-and-tools/managed-agents#memory) — SHA256 versioning, per-store access control, manifest-based discovery.

In [ ]:
import json

memory_dir = f'{PROJECT_DIR}/memory'
stores_info = []

for store_name in sorted(os.listdir(memory_dir)):
    store_path = os.path.join(memory_dir, store_name)
    if not os.path.isdir(store_path):
        continue
    manifest_path = os.path.join(store_path, '_manifest.json')
    if not os.path.exists(manifest_path):
        continue
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    # Count documents (exclude _manifest.json, .v* backups, hidden files)
    docs = [d for d in os.listdir(store_path) 
            if not d.startswith('_') and not d.startswith('.') and not re.search(r'\.v\d+$', d)]
    
    stores_info.append({
        'name': manifest['name'],
        'description': manifest['description'],
        'access': manifest['access'],
        'documents': len(docs),
        'doc_names': docs
    })

table = '| Store | Access | Documents | Description |\n|-------|--------|-----------|-------------|\n'
for s in stores_info:
    table += f'| `{s["name"]}` | {s["access"]} | {s["documents"]} | {s["description"]} |\n'

display(Markdown(f'### Memory Stores ({len(stores_info)})\n\n' + table))

In [ ]:
# Show the patient's health profile — this is what the agent reads at session start
display(Markdown('### Patient Health Profile\n\n'
                 'These documents represent Maria, a 72-year-old managing diabetes, hypertension, and knee arthritis.\n'))

health_profile_dir = f'{PROJECT_DIR}/memory/health-profile'
for doc_name in sorted(os.listdir(health_profile_dir)):
    if doc_name.startswith('_') or re.search(r'\.v\d+$', doc_name):
        continue
    with open(os.path.join(health_profile_dir, doc_name)) as f:
        content = f.read()
    display(Markdown(f'---\n#### `health-profile/{doc_name}`\n\n{content}'))

---
## 3. Run a Care Session with Gemma 4

Now we run the actual CareCompanion agent powered by **Gemma 4 26B-A4B** via OpenRouter. The agent will:

1. Check the current time and calendar
2. Read the patient's health profile from memory
3. Use its 15 tools (phone HAL + skills + memory) to provide care
4. Record a session trace for dream consumption

We run in `--stub` mode, which uses automated responses instead of waiting for user input (suitable for notebook execution).

In [ ]:
%%bash
cd /kaggle/working/skillos_x_mobile

echo "=== Running CareCompanion with Gemma 4 ==="
echo ""

# Run in stub mode (automated, no browser needed)
npx tsx src/main.ts --stub --backend gemma4 \
  --task "Good morning! Please help me with my daily wellness routine. Check my schedule, remind me about medications, and guide me through some exercises." \
  2>&1

echo ""
echo "=== Session complete ==="

---
## 4. Examine the Session Trace

After each session, CareCompanion records a full transcript in `traces/`. This includes:
- YAML frontmatter with metadata (model, duration, skills loaded, memory operations)
- Full multi-turn conversation with tool calls and results

These traces are the input to the Dream Engine.

In [ ]:
import glob

trace_files = sorted(glob.glob(f'{PROJECT_DIR}/traces/*.md'), reverse=True)
print(f'Total session traces: {len(trace_files)}\n')

# Show the most recent trace (the one we just generated)
latest_trace = trace_files[0]
print(f'Latest trace: {os.path.basename(latest_trace)}\n')

with open(latest_trace) as f:
    trace_content = f.read()

# Parse frontmatter
fm_match = re.match(r'^---\s*\n([\s\S]*?)\n---\s*\n([\s\S]*)$', trace_content)
if fm_match:
    frontmatter = fm_match.group(1)
    body = fm_match.group(2)
    
    # Extract metadata
    def extract(key):
        m = re.search(rf'^{key}:\s*"?([^"\n]*)"?', frontmatter, re.M)
        return m.group(1).strip() if m else 'N/A'
    
    meta_table = f"""### Session Metadata

| Field | Value |
|-------|-------|
| **Model** | `{extract('model')}` |
| **Outcome** | {extract('outcome')} |
| **Turns** | {extract('turns')} |
| **Duration** | {int(extract('duration_ms')):,}ms |
| **Skills Loaded** | {extract('skills_loaded')} |
| **Memory Reads** | {extract('memory_reads')} |
| **Memory Writes** | {extract('memory_writes')} |"""
    
    display(Markdown(meta_table))

In [ ]:
# Show the full conversation trace
display(Markdown('### Full Conversation Trace\n'))
display(Markdown(body[:3000] + ('\n\n*... (truncated for display)*' if len(body) > 3000 else '')))

---
## 5. All Session Traces — Dream Input

The repository includes **pre-seeded session traces** spanning 5 days of care interactions. These simulate a realistic care scenario with Maria, covering:
- Morning medication checks
- Evening medication reminders
- Exercise coaching sessions
- Social check-ins after family calls

The Dream Engine will analyze all of these to discover patterns.

In [ ]:
# List all traces with metadata
traces_summary = '| # | Date | Task | Outcome | Turns | Skills | Mem Reads |\n'
traces_summary += '|---|------|------|---------|-------|--------|-----------|\n'

for i, tf in enumerate(sorted(trace_files), 1):
    with open(tf) as f:
        content = f.read()
    fm = re.match(r'^---\s*\n([\s\S]*?)\n---', content)
    if not fm:
        continue
    fm_text = fm.group(1)
    
    def ext(key):
        m = re.search(rf'^{key}:\s*"?([^"\n]*)"?', fm_text, re.M)
        return m.group(1).strip() if m else ''
    
    timestamp = ext('timestamp')[:10]
    task = ext('task')[:50] + ('...' if len(ext('task')) > 50 else '')
    outcome = ext('outcome')
    turns = ext('turns')
    skills = ext('skills_loaded') or 'none'
    reads = ext('memory_reads')
    traces_summary += f'| {i} | {timestamp} | {task} | {outcome} | {turns} | {skills} | {reads} |\n'

display(Markdown(f'### Session Traces ({len(trace_files)} total)\n\n' + traces_summary))

---
## 6. Dreams — Offline Memory Consolidation

The Dream Engine is our most novel contribution. Inspired by biological memory consolidation during sleep, it:

1. **Loads** all existing memory documents
2. **Loads** all session transcripts
3. **Calls Gemma 4** with a consolidation prompt combining existing knowledge + new observations
4. **Reorganizes** knowledge into clean, deduplicated documents
5. **Discovers** clinical insights invisible in any single session

This is functionally equivalent to [Anthropic's Dreams API](https://docs.anthropic.com/en/docs/agents-and-tools/managed-agents#dreams) — implemented with open-weight Gemma 4.

### Running the Dream Engine

In [ ]:
%%bash
cd /kaggle/working/skillos_x_mobile

echo "=== Running Dream Consolidation with Gemma 4 ==="
echo ""

npx tsx src/dream_cli.ts --backend gemma4 2>&1

echo ""
echo "=== Dream complete ==="

---
## 7. Dream Results — Clinical Insights

The dream engine produces two outputs:
1. **Consolidated memory documents** — reorganized, deduplicated knowledge
2. **Clinical insights** — patterns discovered across multiple sessions

These insights are the kind of observations that currently require a dedicated care coordinator reviewing paper notes.

In [ ]:
# Read the dream journal
journal_path = f'{PROJECT_DIR}/memory/consolidated/_dream_journal.md'
if os.path.exists(journal_path):
    with open(journal_path) as f:
        journal = f.read()
    display(Markdown('### Dream Journal\n\n' + journal))
else:
    print('Dream journal not found — dream may not have completed successfully.')

In [ ]:
# Show all consolidated memory documents
consolidated_dir = f'{PROJECT_DIR}/memory/consolidated'

display(Markdown('### Consolidated Memory — Dream Output\n\n'
                 'These documents were produced by the Dream Engine, reorganizing knowledge from all sessions:\n'))

def walk_consolidated(directory, prefix=''):
    """Recursively walk the consolidated store and display documents."""
    for item in sorted(os.listdir(directory)):
        if item.startswith('_') or item.startswith('.') or re.search(r'\.v\d+$', item):
            continue
        full_path = os.path.join(directory, item)
        rel_path = f'{prefix}/{item}' if prefix else item
        if os.path.isdir(full_path):
            walk_consolidated(full_path, rel_path)
        else:
            with open(full_path) as f:
                content = f.read()
            display(Markdown(f'---\n#### `consolidated/{rel_path}`\n\n{content}'))

walk_consolidated(consolidated_dir)

---
## 8. Comparing Input vs. Dream Output

The Dream Engine doesn't just copy — it **reorganizes, deduplicates, and enriches** the data. Let's compare the original medication profile with the dream-consolidated version.

In [ ]:
# Original vs. Consolidated comparison
original_meds = f'{PROJECT_DIR}/memory/health-profile/medications.md'
consolidated_meds = f'{PROJECT_DIR}/memory/consolidated/health-profile/medications.md'

display(Markdown('### Side-by-Side: Medication Records\n'))

if os.path.exists(original_meds):
    with open(original_meds) as f:
        orig = f.read()
    display(Markdown(f'#### Original (`health-profile/medications.md`)\n\n{orig}'))

display(Markdown('---'))

if os.path.exists(consolidated_meds):
    with open(consolidated_meds) as f:
        consol = f.read()
    display(Markdown(f'#### Dream-Consolidated (`consolidated/health-profile/medications.md`)\n\n{consol}'))
else:
    # Check if the dream engine put it at a different path
    for root, dirs, files in os.walk(f'{PROJECT_DIR}/memory/consolidated'):
        for fname in files:
            if 'medication' in fname.lower() and not re.search(r'\.v\d+$', fname):
                fpath = os.path.join(root, fname)
                with open(fpath) as f:
                    consol = f.read()
                rel = os.path.relpath(fpath, f'{PROJECT_DIR}/memory/consolidated')
                display(Markdown(f'#### Dream-Consolidated (`consolidated/{rel}`)\n\n{consol}'))

display(Markdown('\n> **Key difference**: The dream-consolidated version includes *adherence patterns* '
                 'observed across sessions — information that didn\'t exist in the original static profile. '
                 'The Dream Engine connects behavioral observations from individual sessions into actionable clinical trends.'))

---
## 9. Architecture Summary

```
                        CareCompanion Architecture

    ┌─────────────────────────────────────────────────────┐
    │                   Agent Loop                         │
    │  System prompt + Skills metadata + Memory context    │
    │         ↕ generate() with tool definitions           │
    │  ┌─────────────┐  ┌──────────┐  ┌──────────────┐   │
    │  │  Backend     │  │  Tools   │  │  Memory      │   │
    │  │  (Gemma 4)   │  │  (15)    │  │  Stores      │   │
    │  └─────────────┘  └──────────┘  └──────────────┘   │
    │         ↕                ↕              ↕            │
    │  ┌─────────────┐  ┌──────────┐  ┌──────────────┐   │
    │  │  OpenRouter  │  │ MobileHAL│  │  Filesystem  │   │
    │  └─────────────┘  └──────────┘  └──────────────┘   │
    └─────────────────────────────────────────────────────┘
             ↓ (after session)
    ┌─────────────────────────────────────────────────────┐
    │               Session Trace Recorder                 │
    │  YAML frontmatter + full conversation → traces/      │
    └─────────────────────────────────────────────────────┘
             ↓ (offline, async)
    ┌─────────────────────────────────────────────────────┐
    │                  Dream Engine                        │
    │  Reads: memory/ + traces/                            │
    │  Produces: consolidated knowledge + clinical insights│
    │  Writes: memory/consolidated/                        │
    └─────────────────────────────────────────────────────┘
```

### Multi-Backend Support

The same agent code runs identically across three backends:

| Backend | Model | Provider | Use Case |
|---------|-------|----------|----------|
| `gemma4` | Gemma 4 26B-A4B | OpenRouter | Default — demonstrates open-weight parity |
| `gemini` | Gemini 2.5 Flash | OpenRouter | Fast inference, good for demos |
| `anthropic` | Claude Sonnet 4 | Anthropic API | Reference implementation |

---
## 10. Impact & Conclusion

For a single patient over 5 days, CareCompanion:

- **Tracked 20+ medication events**, identifying a 57% evening Metformin adherence gap
- **Detected declining Vitamin D adherence** linked to supply issues
- **Correlated social isolation with medication non-compliance**
- **Discovered afternoon loneliness (2-5 PM)** as the primary mood disruptor
- **Found exercise participation follows a weekly energy cycle**

### Key Finding

> **Gemma 4 26B-A4B can fully replicate Skills, Memory, and Dreams** — three primitives that Anthropic provides as managed services for their proprietary models. The dream engine's ability to discover that *afternoon loneliness drives evening medication non-compliance* — connecting social, emotional, and medical dimensions across multiple sessions — demonstrates the clinical reasoning power of open-weight models.

### Scale This

Scale CareCompanion to millions of patients and you have a system that could:
- Reduce medication-related hospital readmissions
- Provide early warning to caregivers about behavioral changes
- Maintain continuity of care without human note-taking
- Operate on any device, using open-weight models — no vendor lock-in

---

**GitHub**: [EvolvingAgentsLabs/skillos_x_mobile](https://github.com/EvolvingAgentsLabs/skillos_x_mobile)  
**Video**: [YouTube Demo](https://youtu.be/8ho3SpY-rDU)  
**License**: Apache 2.0  
**Team**: [EvolvingAgentsLabs](https://github.com/EvolvingAgentsLabs)